<a href="https://colab.research.google.com/github/swrobuts/e-comm/blob/main/notebooks/sentiment_vader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Sentiment-Analyse mit VADER
### UrbanCart reviews.csv · THWS E-Commerce Kurs · Bonus-Lab

**VADER** (Valence Aware Dictionary and sEntiment Reasoner) ist ein regelbasiertes Sentiment-Tool, das speziell für Social-Media-Texte entwickelt wurde – funktioniert aber auch sehr gut auf E-Commerce Reviews.

**Was wir machen:**
1. `reviews.csv` laden und erkunden
2. VADER-Scores für jeden Review berechnen
3. Ergebnisse mit den originalen Labels vergleichen
4. Sentiment-Verteilung mit Plotly visualisieren
5. Wordcloud aus negativen Reviews erstellen

---
> 💡 **Colab-Tipp:** `reviews.csv` mit dem Ordner-Symbol links hochladen, oder direkt von GitHub laden (Zelle unten).

In [ ]:
# ── Schritt 0: Pakete installieren ──────────────────────────────────────────
!pip install vaderSentiment plotly wordcloud matplotlib --quiet

In [ ]:
# ── Schritt 1: Daten laden ──────────────────────────────────────────────────
import pandas as pd

# Option A: direkt von GitHub laden (kein Upload nötig)
URL = "https://raw.githubusercontent.com/swrobuts/e-comm/main/data/reviews.csv"
df = pd.read_csv(URL)

# Option B: lokale Datei (Colab: links hochladen)
# df = pd.read_csv('reviews.csv')

print(f"✅ Geladen: {len(df):,} Reviews")
print(f"Spalten: {list(df.columns)}")
df.head()

In [ ]:
# ── Schritt 2: Daten verstehen ──────────────────────────────────────────────
print("=== Rating-Verteilung ===")
# Korrektur: Die Spalte heißt 'stars', nicht 'rating'
print(df['stars'].value_counts().sort_index())

print("\n=== Originale Sentiment-Labels ===")
print(df['sentiment'].value_counts())

print("\n=== Beispiel-Reviews ===")
for _, row in df.sample(3, random_state=42).iterrows():
    # Auch hier 'stars' statt 'rating' nutzen
    print(f"  Rating {row['stars']} | {row['sentiment']:8s} | {str(row['review_text'])[:80]}...")

In [ ]:
# ── Schritt 3: VADER-Scores berechnen ───────────────────────────────────────
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

# Compound-Score für jeden Review berechnen
df['vader_compound'] = df['review_text'].astype(str).apply(
    lambda text: analyzer.polarity_scores(text)['compound']
)

# Compound → Label (Standard-Schwellwerte nach Hutto & Gilbert 2014)
def compound_to_label(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df['vader_label'] = df['vader_compound'].apply(compound_to_label)

print("✅ VADER-Scores berechnet")
print("\nVADER-Label-Verteilung:")
print(df['vader_label'].value_counts())

# Vergleich: VADER vs. Original
print("\n=== VADER vs. Original-Label ===")
print(pd.crosstab(df['sentiment'], df['vader_label'], margins=True))

In [ ]:
# ── Schritt 4: Accuracy berechnen ───────────────────────────────────────────
correct = (df['sentiment'] == df['vader_label']).sum()
total = len(df)
accuracy = correct / total * 100

print(f"VADER Accuracy: {correct:,} / {total:,} = {accuracy:.1f}%")
print("\n→ VADER wurde auf Social-Media-Texte trainiert, nicht auf E-Commerce-Reviews.")
print("  Für bessere Ergebnisse: spezialisierte Transformer-Modelle (Notebook 2).")

In [ ]:
# ── Schritt 5: Visualisierung – Compound-Score-Verteilung ───────────────────
import plotly.express as px
import plotly.graph_objects as go

# Histogramm der Compound-Scores
fig1 = px.histogram(
    df, x='vader_compound', nbins=50,
    color='vader_label',
    color_discrete_map={'positive': '#E87722', 'neutral': '#888', 'negative': '#005B9A'},
    title='VADER Compound-Score-Verteilung (UrbanCart Reviews)',
    labels={'vader_compound': 'Compound-Score (-1 = negativ, +1 = positiv)', 'count': 'Anzahl Reviews'},
    template='simple_white'
)
fig1.add_vline(x=0.05, line_dash='dash', line_color='gray', annotation_text='pos. Grenze')
fig1.add_vline(x=-0.05, line_dash='dash', line_color='gray', annotation_text='neg. Grenze')
fig1.show()

In [ ]:
# ── Schritt 6: Sentiment pro Produktkategorie ━━━━━━━━━━━━━━━━
# Mittleren Compound-Score pro Produktkategorie
if 'product_name' in df.columns:
    # Top 10 Produkte nach Anzahl Reviews
    top_products = df['product_name'].value_counts().head(10).index
    df_top = df[df['product_name'].isin(top_products)]

    avg_sentiment = df_top.groupby('product_name')['vader_compound'].mean().sort_values()

    fig2 = px.bar(
        x=avg_sentiment.values,
        y=avg_sentiment.index,
        orientation='h',
        title='Durchschnittlicher VADER-Score pro Produkt (Top 10)',
        labels={'x': 'Ø Compound-Score', 'y': 'Produkt'},
        color=avg_sentiment.values,
        # 'Spectral' bietet einen der sanftesten Verläufe von Rot über Gelb nach Blau
        color_continuous_scale='Spectral',
        # Kalibrierung auf die tatsächlichen Datenwerte für maximalen Kontrast
        range_color=[avg_sentiment.min(), avg_sentiment.max()],
        template='simple_white'
    )
    fig2.update_layout(showlegend=False, coloraxis_showscale=True)
    fig2.show()
else:
    print("Spalte 'product_name' nicht vorhanden – Schritt übersprungen")

In [ ]:
# ── Schritt 7: Wordcloud aus negativen Reviews ───────────────────────────────
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

# Stopwords für Englisch
stopwords = set(STOPWORDS)
stopwords.update(['product', 'item', 'one', 'got', 'get', 'would', 'also', 'really', 'ordered'])

# Negative Reviews filtern (VADER)
negative_text = ' '.join(
    df[df['vader_label'] == 'negative']['review_text'].astype(str).tolist()
)

wc = WordCloud(
    width=900, height=400,
    background_color='white',
    stopwords=stopwords,
    colormap='RdYlBu_r',
    max_words=80,
    prefer_horizontal=0.8
).generate(negative_text)

plt.figure(figsize=(14, 6))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Häufigste Wörter in negativen Reviews (VADER)', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

# Top-10 Wörter
print("\nTop 10 Wörter in negativen Reviews:")
freqs = wc.process_text(negative_text)
for word, freq in sorted(freqs.items(), key=lambda x: -x[1])[:10]:
    print(f"  {word}: {freq}")

## 🎯 Zusammenfassung

| Merkmal | VADER |
|---|---|
| **Typ** | Regelbasiert (Lexikon + Regeln) |
| **Setup** | `pip install vaderSentiment` – sofort einsatzbereit |
| **Geschwindigkeit** | ⚡ Sehr schnell (5.800 Reviews in < 2 Sekunden) |
| **Accuracy** | ~60–70 % auf E-Commerce-Reviews |
| **Stärken** | Kein Training nötig, versteht Großbuchstaben, Satzzeichen, Emojis |
| **Schwächen** | Kennt keine Ironie, domänenspezifische Fachbegriffe fehlen |
| **Ideal für** | Schnelle Exploration, erste Übersicht, Proof of Concept |

**→ Für höhere Genauigkeit:** Notebook 2 – Transformer-Modell (cardiffnlp/twitter-roberta-base-sentiment-latest)